# 🎯 Das Gesamtbild

## Evaluation + Tuning = Die Software der Zukunft

Wir haben jedes Teil einzeln gesehen. Jetzt fügen wir alles zusammen — und am Ende gibt's ein Quiz! 😎

In [ ]:
import sys; sys.path.insert(0, ".")
import dspy, ipywidgets as widgets
from dspy_tasks.tasks import get_task, list_tasks, list_by_tier, TASK_REGISTRY
from dspy_tasks.actions import run_baseline, run_optimization, compare_models
from dspy_tasks.visualize import *
from dspy_tasks.config import get_available_models, get_default_model, configure_dspy

MODELS = get_available_models()

In [ ]:
from dspy_tasks.visualize import diagram, diagram_compare

diagram([
    {"label": "Dein Code", "detail": "Signatures + Metriken + Daten", "icon": "📝", "color": "#0078d4"},
    {"label": "Optimizer", "detail": "probiert Varianten", "icon": "⚙️", "color": "#ca5010"},
    {"label": "Evaluator", "detail": "misst Qualität", "icon": "📐", "color": "#ca5010"},
    {"label": "Ergebnis", "detail": "Optimierter Prompt", "icon": "🎯", "color": "#107c10"},
    {"label": "Produktion", "detail": "save & deploy", "icon": "🚀", "color": "#107c10"},
], title="Die komplette Pipeline")

diagram_compare(
    {"title": "Klassische Software", "items": ["Quellcode", "Compiler", "Binary", "Test-Suite", "CI/CD"], "icon": "💻", "color": "#8a8886"},
    {"title": "KI-Software", "items": ["Signatures + Metriken", "Optimizer", "Optimierter Prompt", "Evaluations-Dataset", "Re-Optimierung"], "icon": "🧠", "color": "#0078d4"},
    title="Der neue Software-Stack"
)

## 🏆 Der Cross-Model Showdown

Jetzt lassen wir repräsentative Tasks aus jeder Tier auf verschiedenen Modellen laufen. Das Ergebnis: eine Heatmap die zeigt, wo welches Modell stark oder schwach ist.

In [ ]:
# Pick representative tasks from each tier
SHOWCASE_TASKS = ["sentiment", "math_word", "ticket_routing", "calculator_agent"]

btn = run_button("Run Full Showdown")
out = widgets.Output()

def on_showdown(b):
    with out:
        out.clear_output()
        task_names = []
        all_scores = []

        for task_id in SHOWCASE_TASKS:
            task = get_task(task_id)
            task_names.append(task.name)
            row = []

            for model in MODELS:
                print(f"  ⏳ {task.name} on {model.split('/')[-1]}...", end=" ")
                result = run_baseline(task_id, model, max_eval=5)
                row.append(result.score)
                print(f"{result.score:.0%}")

            all_scores.append(row)
            print()

        model_short = [m.split("/")[-1] for m in MODELS]
        fig = heatmap_tasks_models(task_names, model_short, all_scores,
            title="Task × Model Performance Matrix")
        fig.show()

btn.on_click(on_showdown)
display(btn, out)

## 💰 Der ROI von Optimierung

Optimierung kostet ein paar Dutzend LLM-Aufrufe einmalig. Aber die Verbesserung gilt für JEDEN zukünftigen Aufruf. Spiel mit den Reglern und sieh den ROI!

In [ ]:
opt_calls = widgets.IntSlider(value=30, min=5, max=100, description="Optimization calls:")
cost_per = widgets.FloatSlider(value=0.003, min=0.001, max=0.01, step=0.001,
                                 description="$/call:", readout_format=".3f")
baseline_acc = widgets.FloatSlider(value=0.65, min=0.1, max=0.95, step=0.05, description="Baseline:")
optimized_acc = widgets.FloatSlider(value=0.88, min=0.1, max=0.99, step=0.05, description="Optimized:")
queries = widgets.IntSlider(value=10000, min=100, max=100000, step=1000, description="Queries:")

roi_out = widgets.Output()

def update_roi(*args):
    with roi_out:
        roi_out.clear_output()
        fig = cost_roi_chart(
            opt_calls.value, cost_per.value,
            baseline_acc.value, optimized_acc.value,
            queries.value, cost_per.value
        )
        fig.show()

for w in [opt_calls, cost_per, baseline_acc, optimized_acc, queries]:
    w.observe(update_roi, 'value')

display(widgets.VBox([opt_calls, cost_per, baseline_acc, optimized_acc, queries]), roi_out)
update_roi()  # initial render

## 🔄 Der neue Software-Stack

| Klassische Software | KI-Software |
|---|---|
| Quellcode | Signatures + Metriken + Daten |
| Compiler | Optimizer (z.B. BootstrapFewShot / MIPROv2) |
| Binary | Optimierter Prompt + Few-Shot Beispiele |
| Test-Suite | Evaluations-Dataset |
| CI/CD | Re-Optimierungs-Pipeline |
| Refactoring | Re-Kompilieren mit neuen Daten/Modell |

## 💾 Produktions-Patterns

In Produktion speicherst du das optimierte Modul mit `module.save('pfad.json')` und lädst es mit `module.load('pfad.json')`. Die Optimierungskosten fallen einmalig an — der Nutzen gilt für jede zukünftige Anfrage.

In [ ]:
display_insight("Speichern & Laden optimierter Module",
    "In Produktion speicherst du das optimierte Modul mit module.save('pfad.json') "
    "und lädst es mit module.load('pfad.json'). Die Optimierungskosten fallen einmalig an; "
    "der Nutzen gilt für jede zukünftige Anfrage.",
    icon="💾")

## 🎯 Die Pointe

In [ ]:
from IPython.display import display, Markdown

display(Markdown("""
---
## 🎯 Die Pointe

> **Evaluation** ist die Spezifikation.
>
> **Optimierung** ist der Compiler.
>
> **Daten** sind der Quellcode.

*Willkommen bei Software 3.0.*

---
"""))

## 🧠 Quiz: Was hast du gelernt?

Jetzt testen wir, was hängen geblieben ist! 7 Fragen zu den wichtigsten Konzepten aus allen Notebooks. Viel Erfolg!

In [ ]:
from dspy_tasks.visualize import quiz

quiz([
    {
        "question": "Was ist eine dspy.Signature?",
        "options": [
            "Ein Prompt den du von Hand schreibst",
            "Eine reine Daten-Deklaration die beschreibt was rein- und rausgeht",
            "Eine Python-Funktion die das LLM aufruft",
            "Ein API-Schlüssel für das Modell",
        ],
        "answer": 1,
        "explanation": "Signatures sind DATA — reine Deklarationen ohne Verhalten (Notebook 01).",
    },
    {
        "question": "Warum sind Metriken 'Calculations' im Sinne von Grokking Simplicity?",
        "options": [
            "Weil sie das LLM aufrufen",
            "Weil sie Seiteneffekte haben",
            "Weil sie reine Funktionen sind: gleicher Input = gleicher Output, kein I/O",
            "Weil sie nur in Jupyter funktionieren",
        ],
        "answer": 2,
        "explanation": "Metriken sind pure Functions — testbar ohne API-Key, deterministisch (Notebook 01).",
    },
    {
        "question": "Was ist der Unterschied zwischen dspy.Predict und dspy.ChainOfThought?",
        "options": [
            "Predict ist schneller und besser",
            "ChainOfThought fügt automatisch einen Reasoning-Schritt hinzu",
            "Es gibt keinen Unterschied",
            "Predict nutzt Tools, ChainOfThought nicht",
        ],
        "answer": 1,
        "explanation": "ChainOfThought ist ein 'tieferes Modul' — gleiches Interface, aber mit Denkschritt (Notebook 02).",
    },
    {
        "question": "Was macht der DSPy Optimizer?",
        "options": [
            "Er schreibt Python-Code",
            "Er sucht automatisch nach besseren Prompts und Few-Shot-Beispielen",
            "Er trainiert das LLM-Modell neu",
            "Er übersetzt Prompts in andere Sprachen",
        ],
        "answer": 1,
        "explanation": "Der Optimizer ist wie ein Compiler: Signature + Metrik + Daten → optimierter Prompt (Notebook 04).",
    },
    {
        "question": "Warum sind 'deine Daten dein Burggraben'?",
        "options": [
            "Weil Daten teuer sind",
            "Weil ein generisches Modell + deine Domain-Daten + Tuning etwas erzeugt, das kein Konkurrent kopieren kann",
            "Weil man ohne Daten kein LLM nutzen kann",
            "Weil Daten in der Cloud gespeichert werden",
        ],
        "answer": 1,
        "explanation": "Domain-spezifisches Tuning mit eigenen Daten schafft einen Wettbewerbsvorteil (Notebook 05).",
    },
    {
        "question": "Was optimiert DSPy bei Agenten (ReAct)?",
        "options": [
            "Nur die Worte im Prompt",
            "Die Geschwindigkeit der API-Aufrufe",
            "Nicht nur den Prompt, sondern auch WIE der Agent Tools einsetzt",
            "Die Anzahl der verfügbaren Tools",
        ],
        "answer": 2,
        "explanation": "DSPy optimiert die Entscheidungsstrategie des Agenten — wann und wie er Tools nutzt (Notebook 06).",
    },
    {
        "question": "Was ist 'Software 3.0'?",
        "options": [
            "Die neueste Version von Python",
            "Evaluation = Spezifikation, Optimierung = Compiler, Daten = Quellcode",
            "Ein neues Betriebssystem",
            "Software die nur mit GPT-4o funktioniert",
        ],
        "answer": 1,
        "explanation": "Software 3.0: Du schreibst Metriken (Specs), der Optimizer kompiliert Prompts, deine Daten sind der Source Code.",
    },
])

## 🎓 Und jetzt?

Du hast die Grundlagen verstanden. Hier sind deine nächsten Schritte:

1. **Eigene Daten** — Bring deine eigenen Datasets mit und optimiere darauf
2. **Neue Tasks** — Erweitere die `dspy_tasks/tasks/` mit eigenen Aufgaben
3. **Andere Modelle** — Probier verschiedene Modelle aus und vergleiche
4. **Produktion** — Speichere optimierte Module und deploye sie

### 📚 Optionale Appendix-Notebooks

- **Appendix A: Grokking Simplicity** — Wie man LLM-Code in Data, Calculations und Actions aufteilt
- **Appendix B: Deep Modules** — Warum `Predict`, `ChainOfThought` und `ReAct` das gleiche Interface haben

*Evaluation ist die Spezifikation. Optimierung ist der Compiler. Daten sind der Quellcode.* 🚀